# DS 8001 — Lab 8: Scalable Indexing

This lab works with both halves of the lecture: **hash functions**, which index keys for *exact-match* lookup, and **permutation-based indexing (PBI)**, which indexes objects for *approximate similarity* search.

In Problems 1 and 2 you implement one small function each and use it to reproduce a result you have already seen in the lecture. Problems 3–5 supply the algorithms and ask you to run the comparisons: how much of a permutation you actually need, how MIF and MPT rank the same objects differently, and what the three distributed schemes cost.

### How to answer written questions

Whenever you are asked for a written prediction, explanation, or comparative analysis, **insert a new Markdown/text cell immediately after the question and type your answer there**. In Google Colab, use **+ Text**; in Jupyter, insert a new cell and change its type to **Markdown**. Do not put prose answers inside code cells or edit the question itself.

## Setup

Run the next cell first. It imports the packages used throughout the lab.

Several experiments below rank objects by *small integer* scores, so ties are frequent. Wherever a ranking is produced, ties are broken at random using an explicitly seeded generator, so re-running a cell always reproduces the same numbers.

In [ ]:
import math
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Problem 1 — Hashing: digit sums, universal families, and the prime modulus

### **Problem Description**

A company stores 16-digit card numbers in a hash table. The lecture compared two ways of turning a key into a slot index:

- the **digit sum**, which adds the 16 digits of the key and therefore always lands in $\{0,\dots,144\}$;
- a member of a **universal family**, which splits the key into four 4-digit blocks $(x_1,x_2,x_3,x_4)$ and computes
$$h_a(x)=\Big(\sum_{i=1}^{4} a_i x_i\Big) \bmod n,$$
with $n$ prime and the coefficients $a_i$ drawn independently and uniformly from $\{0,\dots,n-1\}$.

The theorem of the lecture says that for any two **fixed** distinct keys, $\Pr_a\{h_a(x')=h_a(x'')\}=1/n$. Part A measures what that guarantee is worth in a real table; Part B tests the guarantee itself, and what happens when $n$ is not prime.

### **Provided Code**

The next cell provides the two hash functions and `bucket_stats`, which hashes a set of keys and summarises the buckets that separate chaining would build. Because a lookup has to scan an entire bucket, `keys scanned per lookup` is the quantity a user actually feels. The last two lines re-evaluate the worked example of the lecture slide *Example: evaluating $h_a$*, as a check that the arithmetic is right.

In [ ]:
PRIME = 100103    # the prime modulus used in the lecture example


def digits_to_blocks(key):
    """Split a 16-digit card number into its four 4-digit blocks (x1, x2, x3, x4)."""
    text = f"{key:016d}"
    return [int(text[0:4]), int(text[4:8]), int(text[8:12]), int(text[12:16])]


def digit_sum_hash(key):
    """The lecture's first attempt: add up the 16 digits.
    Whatever the table size, the result always lies in {0, ..., 144}."""
    return sum(int(c) for c in f"{key:016d}")


def make_universal_hash(a, n):
    """Return the single function h_a(x) = (a1*x1 + a2*x2 + a3*x3 + a4*x4) mod n.
    Drawing a uniformly from {0,...,n-1}^4 is what draws h_a from the universal family."""
    def h(key):
        return sum(a_i * x_i for a_i, x_i in zip(a, digits_to_blocks(key))) % n
    return h


def bucket_stats(keys, hash_fn, table_size):
    """Hash every key into a table of `table_size` slots and summarise the buckets that
    separate chaining would create."""
    counts = defaultdict(int)
    for key in keys:
        counts[hash_fn(key) % table_size] += 1          # one bucket per occupied slot

    sizes = np.array(list(counts.values()))
    return {
        "slots used": len(sizes),
        "largest bucket": int(sizes.max()),
        # a bucket of size s contains s(s-1)/2 colliding pairs
        "colliding pairs": int(np.sum(sizes * (sizes - 1) // 2)),
        # a lookup scans its whole bucket, so this is the average lookup cost
        "keys scanned per lookup": float(np.sum(sizes ** 2) / len(keys)),
    }


# Check against the lecture slide: the two card numbers there hash to 50587 and 52773.
h_lecture = make_universal_hash([32217, 99266, 36092, 85585], PRIME)
print(h_lecture(4000123456789010), h_lecture(3510782240372222))

### **Task Description — Part A: collisions in a real table**

Both hash functions are used with the **same** table of $n = 100{,}103$ slots, so the hash function is the only thing that differs.

1. Complete the next cell to call `bucket_stats` for `digit_sum_hash` and for `h_universal`, and collect the two result dictionaries in one DataFrame with one row per hash function.
2. Print the two predictions the theorem makes for the universal family on this table: the expected number of colliding pairs, $\binom{|\text{keys}|}{2}\big/n$, and the expected number of keys scanned per lookup, $1+(|\text{keys}|-1)/n$.

**Written prediction:** before running the cell, add a new Markdown/text cell immediately below this question and predict, in one sentence, roughly how many of the $100{,}103$ slots the digit sum can ever use.

In [ ]:
rng = np.random.default_rng(8001)

# 50,000 distinct 16-digit card numbers, all beginning with a 4.
keys = np.unique(4_000_000_000_000_000 + rng.integers(0, 10**15, size=50_000))
print(f"{len(keys)} distinct keys, table size {PRIME}")

# One draw of the coefficient vector a fixes one member h_a of the universal family.
a = rng.integers(0, PRIME, size=4)
h_universal = make_universal_hash(a, PRIME)

# TODO: call bucket_stats for digit_sum_hash and for h_universal, both with
#       table_size=PRIME, and collect the two dictionaries in one DataFrame
#       with one row per hash function.

# TODO: print the two theoretical predictions for the universal family:
#       expected colliding pairs        = len(keys) * (len(keys) - 1) / 2 / PRIME
#       expected keys scanned per lookup = 1 + (len(keys) - 1) / PRIME

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. Compare the two hash functions on `keys scanned per lookup`, and say how closely the universal family matched the two theoretical predictions. Write 2–3 sentences._

### **Task Description — Part B: where is the randomness?**

The lecture stresses that the randomness in the theorem is **not** in the data: the keys $x'$ and $x''$ are fixed, and the probability is taken over the random draw of $a$. Part B estimates that probability directly.

1. Complete `collision_frequency` in the next cell. It must hold the two keys fixed, draw `trials` independent coefficient vectors $a$ uniformly from $\{0,\dots,n-1\}^4$, and return the fraction of draws for which $h_a(x')=h_a(x'')$.
2. Then complete the second cell to run the four supplied configurations and tabulate the estimate next to $1/n$. All four use keys that differ **only in the last block**, so $b=x_4''-x_4'$ is exactly the quantity the proof had to invert modulo $n$.

In [ ]:
def collision_frequency(n, x_prime, x_second, trials, rng):
    """Estimate Pr_a{h_a(x') = h_a(x'')} for two FIXED keys, each given as a tuple of
    four blocks with 0 <= x_i < n. The randomness is in the draw of a, not in the keys."""
    # TODO: draw a trials-by-4 array of coefficients uniformly from {0, ..., n-1}.
    #       The two keys stay fixed; a is the only random object in this experiment.

    # TODO: evaluate (a1*x1 + ... + a4*x4) mod n for both keys and every drawn a.
    #       Hint: if A is the array of draws, then A @ np.asarray(x_prime) computes
    #       all `trials` dot products at once.

    # TODO: replace the line below by the fraction of draws whose hash values agree.
    return 0.0

In [ ]:
# Each configuration fixes a modulus and two keys differing ONLY in the last block.
configurations = [
    ("prime n = 11",      11, (3, 7, 2, 4), (3, 7, 2, 9)),
    ("composite n = 10",  10, (3, 7, 2, 4), (3, 7, 2, 9)),
    ("composite n = 10",  10, (3, 7, 2, 4), (3, 7, 2, 7)),
    ("prime n = 101",    101, (3, 7, 2, 4), (3, 7, 2, 9)),
]
TRIALS = 40_000
rng = np.random.default_rng(8001)

# TODO: for every configuration, call collision_frequency and build a row holding the
#       label, n, b = x4'' - x4', the estimated collision probability, and 1/n.

# TODO: display the four rows as a DataFrame.

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. Which configuration is far from $1/n$, and which property of $b$ and $n$ explains it? The other composite row does agree with $1/n$ — what does that tell you about using a composite modulus? Write 2–3 sentences._

## Problem 2 — PBI: ordered lists and the Spearman footrule distance

### **Problem Description**

From here on the lecture moves from exact-match lookup to similarity search. PBI encodes an object $o$ by its **pruned ordered list** $\tilde L(o,R)$: the $\tilde n$ pivots of $R$ closest to $o$, nearest first. No distance *value* is ever stored — only the order.

Two encodings are compared with the **Spearman footrule distance**
$$d_{SFD}(q,o_i)=\sum_{r_j \in \tilde L(q,R)\,\cap\,\tilde L(o_i,R)} \big|\tilde L(q,R)_{|r_j}-\tilde L(o_i,R)_{|r_j}\big|,$$
summed over the pivots the two pruned lists have in **common**, because a pruned-away pivot has no rank at all.

### **Provided Code**

The next cell provides `pruned_ordered_list`, which computes $\tilde L(o,R)$ for a real object, and `ranks`, which turns a pruned list into the map $r_j \mapsto \tilde L(o,R)_{|r_j}$. It also hard-codes the running example of the lecture ($N=10$, $n=4$, $\tilde n=2$, with the pivots labelled $1,\dots,4$), so that your implementation can be checked against a result you have already seen.

In [ ]:
def pruned_ordered_list(obj, pivots, n_tilde):
    """L~(o, R): the ids of the n_tilde pivots closest to `obj`, nearest first.
    The stable sort breaks distance ties by pivot index, as the lecture specifies."""
    distances = np.linalg.norm(pivots - obj, axis=1)
    return tuple(int(j) for j in np.argsort(distances, kind="stable")[:n_tilde])


def ranks(plist):
    """{pivot: its 1-based position in plist}, i.e. L~(o,R)_|r for every surviving pivot."""
    return {pivot: position for position, pivot in enumerate(plist, start=1)}


# The running example of the lecture: N = 10 objects, n = 4 pivots, n_tilde = 2.
# These are exactly the pruned lists on the slide "MIF search: worked example".
q_example = (4, 3)
example_lists = {
    1: (3, 4), 2: (3, 2), 3: (3, 4), 4: (2, 3),  5: (2, 1),
    6: (2, 3), 7: (1, 2), 8: (4, 1), 9: (1, 4), 10: (4, 1),
}

### **Task Description**

1. Complete `d_sfd` in the next cell. It should work for pivot labels of any kind, since the running example labels its pivots $1,\dots,4$ while the rest of the lab uses array indices.
2. Then complete the second cell to build a table with one row per object $o_1,\dots,o_{10}$ of the running example, giving its accumulator value $d_{SFD}(q,o_i)$ and the number of pivots it shares with $q$, and print the objects with the smallest accumulator. Your accumulator column should reproduce the table on the lecture slide *MIF search: worked example*.

In [ ]:
def d_sfd(query_list, object_list):
    """Spearman footrule distance between two pruned ordered lists, summed over the
    pivots the two lists have in COMMON."""
    query_ranks = ranks(query_list)
    object_ranks = ranks(object_list)

    # TODO: replace the line below by the sum of |query rank - object rank| over the
    #       pivots that occur in both lists.
    return 0

In [ ]:
# TODO: build a DataFrame with one row per object o1..o10 of the running example,
#       holding its accumulator value d_sfd(q_example, example_lists[i]) and the number
#       of pivots it shares with q, then display it.

# TODO: print the objects whose accumulator value is the smallest.

**Your written answer:**  
_Add a new Markdown/text cell immediately below this question. Both $o_4$ and $o_5$ end with accumulator $0$, but $o_4$ shares a pivot with $q$ while $o_5$ shares none. State what the accumulator fails to record, and why that matters for the final ranking. Write 1–2 sentences._

## Problem 3 — How much of the permutation do you actually need?

### **Problem Description**

$\tilde n$ is the main knob of PBI: it fixes how much of each permutation is stored, and therefore the length of the inverted lists and the cost of a query. On the running example, the lecture's concept check showed that a small $\tilde n$ can push the true nearest neighbour to the *bottom* of the ranking. This problem measures that effect on a database.

We use PBI the way a system does: as a **filter** that returns a short candidate list, which a later step re-ranks with true distances (the DDC step of the PP-index and the merge step of the index-based distributed scheme both do exactly this). The quality measure is therefore
$$\text{recall@}C=\frac{\big|\{\text{true }K\text{-NN of }q\}\ \cap\ \{\text{top }C\text{ candidates}\}\big|}{K},$$
with $K=10$ true neighbours and a candidate list of $C=100$ objects out of $N=2000$.

### **Provided Code**

The first cell below builds the data: $N=2000$ objects in $m=8$ dimensions drawn from six Gaussian clusters, $50$ query objects drawn the same way, and a pivot set $R$ of $n=32$ objects sampled from $D$, as the lecture prescribes. It also provides `exact_knn` (the exhaustive baseline, which computes all $N$ distances per query) and `recall_at_c`, and stores the exact answers in `ground_truth`.

The second cell is the PBI machinery of the lecture: `build_index` (the indexing algorithm), `build_mif` (the metric inverted file) and `mif_search` (the accumulator search, which reads **only** the inverted lists of the query's pivots). Besides the candidate list, `mif_search` returns two diagnostics: how many postings it read, and how many objects had their accumulator never touched because they share no pivot with $q$. `common_pivots` reports how many pivots given objects share with the query.

In [ ]:
rng = np.random.default_rng(8001)

N, m = 2000, 8                  # N objects in m-dimensional Euclidean space
N_CLUSTERS, N_QUERIES = 6, 50
n = 32                          # n = |R| pivots, sampled from D as in the lecture
K, C = 10, 100                  # K true neighbours wanted; PBI returns C candidates

centres = rng.normal(0, 1.5, size=(N_CLUSTERS, m))
D = centres[rng.integers(0, N_CLUSTERS, N)] + rng.normal(size=(N, m))
queries = centres[rng.integers(0, N_CLUSTERS, N_QUERIES)] + rng.normal(size=(N_QUERIES, m))

pivots = D[rng.choice(N, n, replace=False)]      # R is a random sample of D


def exact_knn(objects, q, K):
    """Exhaustive search: compute all N distances and return the K closest object ids."""
    return np.argsort(np.linalg.norm(objects - q, axis=1), kind="stable")[:K]


def recall_at_c(candidate_ids, true_ids):
    """Fraction of the true K nearest neighbours that appear in the candidate list."""
    return len(set(map(int, candidate_ids)) & set(map(int, true_ids))) / len(true_ids)


ground_truth = [exact_knn(D, q, K) for q in queries]
print(f"N={N}, m={m}, n=|R|={n}, K={K}, C={C}, {N_QUERIES} queries")
print(f"exhaustive search costs {N} distance computations per query")

In [ ]:
def top_k(scores, k, rng, largest=False):
    """Rank objects by `scores` and return the ids of the best k.
    Scores are small integers, so ties are frequent; they are broken at RANDOM so that
    the object id does not silently become part of the ranking."""
    scores = np.asarray(scores, dtype=float)
    tie = rng.random(len(scores))
    return np.lexsort((tie, -scores if largest else scores))[:k]


def build_index(objects, pivots, n_tilde):
    """PBI indexing algorithm: one pruned ordered list per object."""
    distances = np.linalg.norm(objects[:, None, :] - pivots[None, :, :], axis=2)
    order = np.argsort(distances, axis=1, kind="stable")[:, :n_tilde]
    return [tuple(int(j) for j in row) for row in order]


def build_mif(index, n_pivots):
    """Metric inverted file: one inverted list per pivot, holding a posting
    (object id, rank of that pivot) for every object whose pruned list names it."""
    mif = {j: [] for j in range(n_pivots)}
    for object_id, plist in enumerate(index):
        for rank, pivot in enumerate(plist, start=1):
            mif[pivot].append((object_id, rank))
    return mif


def mif_search(mif, query_list, N, C, rng):
    """The accumulator search of the lecture. Only the inverted lists of the query's own
    pivots are read; each posting adds |rank in query list - rank in object list| to the
    object's accumulator. Objects that are never mentioned keep an accumulator of 0.

    Returns the C best candidates, the number of postings read, and the number of
    objects whose accumulator was never touched."""
    accumulator = np.zeros(N)
    touched = np.zeros(N, dtype=bool)
    postings_read = 0

    for rank_q, pivot in enumerate(query_list, start=1):
        for object_id, rank_o in mif[pivot]:
            accumulator[object_id] += abs(rank_q - rank_o)
            touched[object_id] = True
            postings_read += 1

    return top_k(accumulator, C, rng), postings_read, int((~touched).sum())


def common_pivots(index, query_list, object_ids):
    """How many pivots each of the listed objects shares with the query."""
    query_pivots = set(query_list)
    return np.array([len(query_pivots & set(index[int(i)])) for i in object_ids])

### **Task Description**

1. Complete the next cell so that, for every $\tilde n$ in `n_tilde_values`, it loops over all $50$ queries and records the average of four quantities: `recall@100`, postings read, objects sharing no pivot with $q$, and how many pivots the returned top-$100$ candidates share with $q$ (use `common_pivots(...).mean()`).
2. Display the table, then plot `recall@100` against $\tilde n$ and postings read against $\tilde n$ side by side with `plt.subplots(1, 2)`; a logarithmic $y$-axis helps for the postings.
3. Compare the postings read at $\tilde n=32$ with the $N=2000$ distances an exhaustive search needs per query.

**Written prediction:** before running the cell, add a new Markdown/text cell below this question and predict in one sentence whether recall should rise or fall as $\tilde n$ grows.

In [ ]:
n_tilde_values = [2, 4, 8, 16, 24, 32]
tie_rng = np.random.default_rng(2026)

rows = []
for n_tilde in n_tilde_values:
    index = build_index(D, pivots, n_tilde)
    mif = build_mif(index, n)

    # TODO: for every query, build L~(q, R) with pruned_ordered_list, run mif_search,
    #       and collect recall_at_c(candidates, true_ids), the postings read, the number
    #       of objects sharing no pivot, and common_pivots(index, query_list, candidates).mean().

    # TODO: append one row holding n_tilde and the four averages.

# TODO: display the table.

# TODO: plot recall against n_tilde and postings read against n_tilde side by side.

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. Recall is very low for the smallest values of $\tilde n$. Use the last two columns of your table to explain what kind of object the accumulator is putting at the top of the candidate list, and connect this to what you observed in Problem 2. Write 2–3 sentences._

## Problem 4 — Repairing the accumulator: the metric permutation table

### **Problem Description**

MPT keeps the same index and changes how a pruned list is *scored*. Ranks are first quantised into $B\le\tilde n$ buckets,
$$b_{ij}=\Big\lceil \tfrac{B}{\tilde n}\,\tilde L(o_i,R)_{|r_j}\Big\rceil,\qquad b_{qj}=\Big\lceil \tfrac{B}{\tilde n}\,\tilde L(q,R)_{|r_j}\Big\rceil,$$
and objects are then ranked by **decreasing**
$$s_i=\big|\{\,r_j\in \tilde L(q,R)\ :\ r_j\in \tilde L(o_i,R)\ \text{ and }\ |b_{ij}-b_{qj}|\le 1\,\}\big|,$$
which **counts matching pivots** instead of adding up rank disagreements. This problem tests whether that repairs the behaviour you diagnosed in Problem 3.

### **Provided Code**

The next cell provides `mpt_bucket`, `mpt_score` and `mpt_search`. For clarity `mpt_search` scores every object directly rather than scanning buckets; this gives the same ranking, because an object that shares no pivot with $q$ sits in no bucket the query reads and therefore scores $0$ either way.

In [ ]:
def mpt_bucket(rank, n_tilde, B):
    """Quantise a rank into one of B buckets: ceil(B / n_tilde * rank)."""
    return math.ceil(B / n_tilde * rank)


def mpt_score(query_list, object_list, n_tilde, B):
    """MPT score s_i: the number of pivots shared by q and o_i whose quantised buckets
    differ by at most 1. Objects are ranked by DECREASING s_i."""
    query_ranks = ranks(query_list)
    object_ranks = ranks(object_list)

    score = 0
    for pivot, rank_q in query_ranks.items():
        if pivot in object_ranks:
            b_q = mpt_bucket(rank_q, n_tilde, B)
            b_i = mpt_bucket(object_ranks[pivot], n_tilde, B)
            if abs(b_i - b_q) <= 1:
                score += 1
    return score


def mpt_search(index, query_list, n_tilde, B, C, rng):
    """Score every object and return the C highest-scoring candidates."""
    scores = np.array([mpt_score(query_list, plist, n_tilde, B) for plist in index])
    return top_k(scores, C, rng, largest=True), scores

### **Task Description**

1. First check the implementation against the lecture. Complete the next cell to tabulate $s_i$ for the ten objects of the running example with $\tilde n=2$ and $B=\tilde n=2$; the values should match the slide *MPT example*.
2. Then complete the second cell to compare the two rankings on the database of Problem 3. For every $\tilde n$ in `compare_values`, run **both** `mif_search` and `mpt_search` on the same index, the same queries and the same candidate-list size $C$, and record the two mean recalls. Use $B=\tilde n$, i.e. no quantisation, so that the scoring rule is the only difference between the two methods.
3. Plot the two recall curves against $\tilde n$ on the same figure.

In [ ]:
# TODO: build and display a DataFrame with one row per object o1..o10 of the running
#       example, holding mpt_score(q_example, example_lists[i], 2, 2).

In [ ]:
compare_values = [2, 4, 8, 16, 32]
tie_rng = np.random.default_rng(2026)

rows = []
for n_tilde in compare_values:
    index = build_index(D, pivots, n_tilde)
    mif = build_mif(index, n)

    # TODO: for every query, build L~(q, R) once and run BOTH mif_search and
    #       mpt_search (with B = n_tilde) on it, collecting the two recalls.

    # TODO: append one row with n_tilde and the two mean recalls.

# TODO: display the table and plot the two recall curves against n_tilde.

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. Explain why MPT is so much better than the MIF accumulator at small $\tilde n$, and why the ordering of the two methods reverses at $\tilde n=n=32$. Write 2–3 sentences._

## Problem 5 — Distributing PBI over $P$ processes

### **Problem Description**

The lecture gave three ways of spreading PBI over $P$ processes and described them as points on one trade-off curve between *communication while indexing* and *query fan-out*:

- **Data division** — $D$ split at random, $R$ replicated everywhere. No indexing communication, but every query must reach all $P$ processes.
- **Reference division** — $D$ split, but each process *owns* $n/P$ pivots and stores their postings, which the indexers must ship to it. A query then wakes only the owners of the pivots in $\tilde L(q,R)$.
- **Index-based division** — $D$ split and each process picks its *own* pivots from its *own* objects. No indexing communication at all, but the local scores are incomparable, so every woken process must re-rank its best $\Delta K$ candidates with **true distances** before the broker can merge them.

All three simulations below score with MPT ($\tilde n=4$, $B=\tilde n$), because $s_i$ is a sum over pivots and can therefore be accumulated from partial results arriving from several processes.

### **Provided Code**

The first cell provides the two schemes that share the pivot set $R$. Note what each one counts: `index distances / process` is the indexing work of the busiest process, `postings sent when indexing` counts only postings that cross a process boundary, and `processes woken / query` is measured rather than assumed. Because both schemes rank with the same global $R$, the broker can add up partial scores directly.

In [ ]:
def random_split(N, P, rng):
    """Split the object ids of D at random into P subdomains of (nearly) equal size."""
    return np.array_split(rng.permutation(N), P)


def local_mpt_scores(local_index, query_list, n_tilde, B):
    """MPT score of every object held by one process."""
    return np.array([mpt_score(query_list, plist, n_tilde, B) for plist in local_index])


def simulate_data_division(D, pivots, queries, ground_truth, n_tilde, B, C, P, rng):
    """R is replicated on every process and D is split at random. Each process indexes
    only its own objects, but against the full pivot set, so nothing is communicated
    while indexing. At query time the broker broadcasts q to ALL P processes."""
    N_local, n_pivots = len(D), len(pivots)
    parts = random_split(N_local, P, rng)
    local_index = [build_index(D[ids], pivots, n_tilde) for ids in parts]

    recalls = []
    for q, true_ids in zip(queries, ground_truth):
        query_list = pruned_ordered_list(q, pivots, n_tilde)     # every process can build this

        scores = np.zeros(N_local)                               # the broker's merged list
        for ids, index_p in zip(parts, local_index):
            scores[ids] = local_mpt_scores(index_p, query_list, n_tilde, B)

        recalls.append(recall_at_c(top_k(scores, C, rng, largest=True), true_ids))

    return {"index distances / process": max(len(ids) * n_pivots for ids in parts),
            "postings sent when indexing": 0,
            "processes woken / query": float(P),
            "true distances / query": 0.0,
            "recall@100": float(np.mean(recalls))}


def simulate_reference_division(D, pivots, queries, ground_truth, n_tilde, B, C, P, rng):
    """D is split and the pivots are partitioned: process p OWNS R_p and stores its
    postings. Every indexer still needs all of R to build its pruned lists, and ships
    each posting to the owner of that pivot. A query wakes only the owners of the
    pivots in L~(q,R)."""
    N_local, n_pivots = len(D), len(pivots)
    parts = random_split(N_local, P, rng)

    owner = np.empty(n_pivots, dtype=int)                        # which process owns each pivot
    for p, owned in enumerate(np.array_split(np.arange(n_pivots), P)):
        owner[owned] = p

    store = [dict() for _ in range(P)]                           # store[p][pivot] = postings
    messages = 0
    for indexer, ids in enumerate(parts):
        for global_id, plist in zip(ids, build_index(D[ids], pivots, n_tilde)):
            for rank, pivot in enumerate(plist, start=1):
                store[owner[pivot]].setdefault(pivot, []).append((int(global_id), rank))
                if owner[pivot] != indexer:
                    messages += 1                                # this posting crosses the network

    recalls, woken_counts = [], []
    for q, true_ids in zip(queries, ground_truth):
        query_list = pruned_ordered_list(q, pivots, n_tilde)     # built by the broker

        scores = np.zeros(N_local)
        woken = set()
        for pivot, rank_q in ranks(query_list).items():
            p = int(owner[pivot])
            woken.add(p)                                         # only this owner is notified
            b_q = mpt_bucket(rank_q, n_tilde, B)
            for global_id, rank_o in store[p].get(pivot, []):
                if abs(mpt_bucket(rank_o, n_tilde, B) - b_q) <= 1:
                    scores[global_id] += 1                       # partial score, summed by the broker

        recalls.append(recall_at_c(top_k(scores, C, rng, largest=True), true_ids))
        woken_counts.append(len(woken))

    return {"index distances / process": max(len(ids) * n_pivots for ids in parts),
            "postings sent when indexing": messages,
            "processes woken / query": float(np.mean(woken_counts)),
            "true distances / query": 0.0,
            "recall@100": float(np.mean(recalls))}

The second cell provides the index-based scheme. Each process draws its own $n/P$ pivots from its own objects, and can keep at most that many, so locally $\tilde n_p=\min(\tilde n,\ n/P)$. The broker knows the union $R=R_1\cup\dots\cup R_P$ and wakes the owners of the pivots in $\tilde L(q,R)$; each woken process returns **true** distances for its best $\Delta K$ candidates, and the broker merges on those distances.

One caveat when reading its numbers: this scheme draws its own pivots, so its row is never exactly comparable with the other two, even at $P=1$. Read its trend across $P$ rather than any single value.

In [ ]:
def simulate_index_division(D, queries, ground_truth, n_total, n_tilde, B, C, K, P, Delta, rng):
    """D is split and each process selects its OWN pivots from its OWN objects, so there
    is no indexing communication at all. The price is that the local scores are not
    comparable: every woken process re-ranks its best Delta*K candidates with true
    distances, and the broker merges on those."""
    N_local = len(D)
    parts = random_split(N_local, P, rng)

    n_local = n_total // P                          # each process owns n/P pivots
    n_tilde_local = min(n_tilde, n_local)           # it cannot keep more pivots than it owns
    local_pivots = [D[rng.choice(ids, n_local, replace=False)] for ids in parts]
    local_index = [build_index(D[ids], R_p, n_tilde_local)
                   for ids, R_p in zip(parts, local_pivots)]

    all_pivots = np.vstack(local_pivots)            # the broker's view of R = R_1 u ... u R_P
    owner = np.repeat(np.arange(P), n_local)

    recalls, woken_counts, true_distances = [], [], []
    for q, true_ids in zip(queries, ground_truth):
        global_list = pruned_ordered_list(q, all_pivots, n_tilde)
        woken = sorted({int(owner[pivot]) for pivot in global_list})

        candidate_ids, candidate_dist, computed = [], [], 0
        for p in woken:
            local_query = pruned_ordered_list(q, local_pivots[p], n_tilde_local)
            scores = local_mpt_scores(local_index[p], local_query, n_tilde_local, B)

            chosen = parts[p][top_k(scores, Delta * K, rng, largest=True)]
            candidate_ids.append(chosen)
            candidate_dist.append(np.linalg.norm(D[chosen] - q, axis=1))   # the DDC step
            computed += len(chosen)

        candidate_ids = np.concatenate(candidate_ids)
        candidate_dist = np.concatenate(candidate_dist)
        best = np.argsort(candidate_dist, kind="stable")[:C]               # merge on true distances

        recalls.append(recall_at_c(candidate_ids[best], true_ids))
        woken_counts.append(len(woken))
        true_distances.append(computed)

    return {"index distances / process": max(len(ids) * n_local for ids in parts),
            "postings sent when indexing": 0,
            "processes woken / query": float(np.mean(woken_counts)),
            "true distances / query": float(np.mean(true_distances)),
            "recall@100": float(np.mean(recalls))}

### **Task Description — Part A: what each scheme costs**

1. Complete the next cell to call the three simulators for every $P$ in `P_values` and collect one row per (scheme, $P$) pair, with the scheme name and $P$ alongside the returned measurements.
2. Display the table, then plot `processes woken / query` against $P$ for the three schemes on one figure.

Pass a **freshly seeded** generator, `np.random.default_rng(2026)`, to every call, so that all three schemes see the same random split of $D$ and the numbers are reproducible.

In [ ]:
N_TILDE_D, B_D, DELTA = 4, 4, 10
P_values = [1, 2, 4, 8]

rows = []
for P in P_values:
    # TODO: call the three simulators for this P, each with np.random.default_rng(2026),
    #       and append one row per scheme holding {"scheme": ..., "P": P, **result}.
    #
    #   simulate_data_division(D, pivots, queries, ground_truth, N_TILDE_D, B_D, C, P, rng)
    #   simulate_reference_division(D, pivots, queries, ground_truth, N_TILDE_D, B_D, C, P, rng)
    #   simulate_index_division(D, queries, ground_truth, n, N_TILDE_D, B_D, C, K, P, DELTA, rng)
    pass

# TODO: display the table.

# TODO: plot "processes woken / query" against P, one line per scheme.

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. Data division and reference division report the same recall at every $P$ — say why that must be so. Then name what reference division pays for that, and what it gets back. Write 2–3 sentences._

### **Task Description — Part B: the price of incomparable scores**

Index-based division is the only scheme that recomputes true distances at query time, and $\Delta$ controls how many. Complete the next cell to run `simulate_index_division` at $P=4$ for every $\Delta$ in `delta_values`, and tabulate `true distances / query` next to `recall@100`.

In [ ]:
delta_values = [1, 2, 5, 10, 20]

rows = []
for Delta in delta_values:
    # TODO: run simulate_index_division at P = 4 with this Delta and a freshly seeded
    #       np.random.default_rng(2026), then append a row with Delta,
    #       "true distances / query" and "recall@100".
    pass

# TODO: display the table.

**Your written answer:**  
_Add a new Markdown/text cell immediately below this question. Your Part A table shows index division's recall falling as $P$ grows, even though it indexes far faster than the other two. The lecture notes that $D$ may be divided by a clustering algorithm instead of at random — explain in 1–2 sentences why that would help this scheme in particular._

## Final takeaway

The two halves of the lecture index for different questions, but both replace an impossible amount of work with a cheap proxy, and both pay for it in a way you can measure.

Hashing replaces a $10^{16}$-slot array with $10^5$ slots and pays in collisions — and Problem 1 showed that the price stays small only when the hash function is drawn from a universal family with a **prime** modulus. PBI replaces $N$ distance computations with a comparison of pivot *orderings* and pays in recall. Problems 3 and 4 showed that this price depends on two separate choices: how much of the permutation is kept ($\tilde n$), and how the kept part is scored — the same index, ranked by the MPT score rather than by the MIF accumulator, was worth more than a fourfold increase in $\tilde n$. Problem 5 showed that distributing the index does not remove that trade-off but adds a second one, between what has to be communicated while indexing and how much of the cluster each query has to wake.